# RouteMinds XGBoost Baseline

This notebook is the notebook-first workflow for the RouteMinds baseline.

- raw source dataset: stop-event simulation Parquet
- smoke check target: stop-level `delay_minutes`
- canonical target: segment-level `actual_segment_minutes`
- routing weight: predicted segment travel time


In [ ]:
from training.config import load_training_config
from training.data import take_group_row_budget
from training.train_xgboost import (
    prepare_datasets,
    run_experiment,
    save_experiment_artifacts,
)

config = load_training_config("training/config/default_config.toml")
config


In [ ]:
raw_df, stop_smoke_df, segment_df = prepare_datasets(config)

print("raw rows:", len(raw_df))
print("stop smoke rows:", len(stop_smoke_df))
print("segment rows:", len(segment_df))

raw_df.head()


In [ ]:
segment_df[[
    "trip_id",
    "from_stop_id",
    "to_stop_id",
    "scheduled_segment_minutes",
    "actual_segment_minutes",
    "segment_delay_minutes",
    "prev_segment_delay",
    "rolling_segment_delay_3",
]].head()


In [ ]:
smoke_df = take_group_row_budget(
    stop_smoke_df,
    config.split.group_by,
    "trip_start_scheduled_unix",
    config.smoke.sample_rows,
)

smoke_result = run_experiment(
    smoke_df,
    config,
    config.stop_smoke_features,
    config.targets.smoke_target,
)

smoke_result.validation_metrics, smoke_result.test_metrics


In [ ]:
canonical_result = run_experiment(
    segment_df,
    config,
    config.segment_features,
    config.targets.canonical_target,
    secondary_target_column=config.targets.secondary_target,
)

canonical_result.validation_metrics, canonical_result.validation_secondary_metrics, canonical_result.test_metrics


In [ ]:
save_experiment_artifacts(
    config,
    raw_dataframe=raw_df,
    segment_dataframe=segment_df,
    canonical_result=canonical_result,
    smoke_result=smoke_result,
)


Run this notebook with the `route_minds` Conda environment so the notebook kernel and the CLI training entrypoint use the same dependencies and model code.